# 02: Transform an Existing Block Model Attribute

This tutorial signs in to Evo, selects a workspace during sign-in, loads an existing block model, and transforms a numeric attribute. Demo 1 publishes the result as a new attribute; Demo 2 replaces an existing attribute in a new block model version.

## Before You Start

You need an Evo app client ID and redirect URL. You also need permission to view and update the selected block model. Each operation creates a new block model version, so the source version is not changed.

## Learning Path

1. [01: Create and Query a Regular Block Model](01-create-and-query-regular-block-model.ipynb)
2. **This notebook**: transform an existing numeric attribute
3. [03: Block Model Reports](03-block-model-reports.ipynb)

In [ ]:
from evo.notebooks import ServiceManagerWidget

# Evo app credentials
client_id = "daves-evo-client"  # Replace with your client ID
redirect_url = "<your-redirect-url>"  # Replace with your redirect URL

# The sign-in flow includes workspace selection.
manager = await ServiceManagerWidget.with_auth_code(
    client_id=client_id,
    # redirect_url=redirect_url,
).login()

### Demo 1: Create a new attribute based on an existing attribute

Choose a block model from the dropdown. The widget lists block models in the workspace selected during sign-in.

In [ ]:
from evo.blockmodels import BlockModelAPIClient

from helpers import BlockModelSelectorWidget

environment = manager.get_environment()

service_client = BlockModelAPIClient(environment, manager.get_connector(), manager.cache)
block_model_selector = await BlockModelSelectorWidget.create(service_client)
display(block_model_selector)

Choose an existing attribute and enter a unique name for the new attribute.

In [ ]:
from evo.objects.data import ObjectReference
from evo.objects.typed import BlockModel

from helpers import BlockModelAttributeSelectorWidget

api_block_model = await service_client.get_block_model(block_model_selector.value)
object_id = api_block_model.geoscience_object_id
if object_id is None:
    raise RuntimeError("The selected block model is not linked to a geoscience object.")

object_reference = ObjectReference.new(environment, object_id=object_id)
block_model = await BlockModel.from_reference(manager, object_reference)

print(f"Selected block model: {block_model.name}")
print(f"Block model ID: {block_model.block_model_uuid}")
attribute_selector = BlockModelAttributeSelectorWidget(block_model.attributes)
display(attribute_selector)

Create the new attribute by transforming the existing attribute. This example applies $new = old \times scale + offset$. Keep the scale at `1.0` and the offset at `0.0` to make an exact copy.

The notebook verifies that the selected attribute is numeric before it is transformed. Use a unit compatible with the transformed values, or leave `new_column_unit` as `None` when no unit is required.

In [ ]:
import pandas as pd

source_attribute = attribute_selector.value
new_attribute = attribute_selector.new_attribute
scale = 2.0
offset = 0.0
new_column_unit = None  # For example: "g/t"

if not new_attribute:
    raise ValueError("Enter a name for the transformed attribute.")
if source_attribute in {"x", "y", "z"}:
    raise ValueError("Choose a block model attribute, not a geometry column.")
if block_model.get_attribute(source_attribute) is None:
    raise ValueError(f"{source_attribute!r} is not an attribute on the selected block model.")
if block_model.get_attribute(new_attribute) is not None:
    raise ValueError(f"{new_attribute!r} already exists. Choose a new attribute name.")

source_data = await block_model.to_dataframe(columns=["x", "y", "z", source_attribute])
if not pd.api.types.is_numeric_dtype(source_data[source_attribute]):
    raise TypeError(f"{source_attribute!r} must contain numeric values.")

comparison_data = source_data.copy()
comparison_data[new_attribute] = source_data[source_attribute] * scale + offset
new_attribute_data = comparison_data[["x", "y", "z", new_attribute]]

display(comparison_data.head())
print(f"Transform: {new_attribute} = {source_attribute} * {scale} + {offset}")

Publish the new attribute. Evo creates a new block model version containing the new attribute.

In [ ]:
version = await block_model.add_attribute(
    data=new_attribute_data,
    attribute_name=new_attribute,
    unit=new_column_unit,
)

print(f"Published {new_attribute!r} in version: {version.version_id}")

### Demo 2: Transform an existing attribute in place

Choose a block model from the dropdown. The widget lists block models in the workspace selected during sign-in.

In [ ]:
from evo.blockmodels import BlockModelAPIClient

from helpers import BlockModelSelectorWidget

environment = manager.get_environment()

service_client = BlockModelAPIClient(environment, manager.get_connector(), manager.cache)

in_place_block_model_selector = await BlockModelSelectorWidget.create(service_client)
display(in_place_block_model_selector)

Choose the attribute to transform. This widget only selects the existing attribute because the transformed values replace that attribute in a new block model version.

In [ ]:
from evo.objects.data import ObjectReference
from evo.objects.typed import BlockModel

from helpers import ExistingBlockModelAttributeSelectorWidget

api_block_model = await service_client.get_block_model(in_place_block_model_selector.value)
object_id = api_block_model.geoscience_object_id
if object_id is None:
    raise RuntimeError("The selected block model is not linked to a geoscience object.")

object_reference = ObjectReference.new(environment, object_id=object_id)
in_place_block_model = await BlockModel.from_reference(manager, object_reference)

print(f"Selected block model: {in_place_block_model.name}")
print(f"Block model ID: {in_place_block_model.block_model_uuid}")
in_place_attribute_selector = ExistingBlockModelAttributeSelectorWidget(in_place_block_model.attributes)
display(in_place_attribute_selector)

Transform the selected numeric attribute. This example applies $updated = original \times scale + offset$. The preview shows the original and transformed values before the attribute is replaced.

An in-place update must retain the attribute's existing data type. For integer attributes, use values that transform to whole numbers.

In [ ]:
import pandas as pd

attribute_to_update = in_place_attribute_selector.value
scale = 2.0
offset = 0.0

if attribute_to_update in {"x", "y", "z"}:
    raise ValueError("Choose a block model attribute, not a geometry column.")
if in_place_block_model.get_attribute(attribute_to_update) is None:
    raise ValueError(f"{attribute_to_update!r} is not an attribute on the selected block model.")

source_data = await in_place_block_model.to_dataframe(
    columns=["x", "y", "z", attribute_to_update]
)
source_values = source_data[attribute_to_update]
if not pd.api.types.is_numeric_dtype(source_values):
    raise TypeError(f"{attribute_to_update!r} must contain numeric values.")

updated_values = source_values * scale + offset
if pd.api.types.is_integer_dtype(source_values):
    if not updated_values.dropna().mod(1).eq(0).all():
        raise ValueError(
            f"{attribute_to_update!r} is an integer attribute, so its transformed values must be whole numbers."
        )
    updated_values = updated_values.astype(source_values.dtype)

updated_attribute_data = source_data.copy()
updated_attribute_data[attribute_to_update] = updated_values
preview_data = source_data.copy()
preview_data[f"{attribute_to_update}_updated"] = updated_values

display(preview_data.head())
print(f"Transform: {attribute_to_update} = {attribute_to_update} * {scale} + {offset}")

Publish the transformed values. Evo replaces the selected attribute in a new block model version; the source version remains unchanged.

The replacement data must retain the selected attribute's existing data type. In particular, integer attributes can only be updated with whole-number values.

In [ ]:
version = await in_place_block_model.update_attributes(
    data=updated_attribute_data,
    update_columns={attribute_to_update},
)

print(f"Updated {attribute_to_update!r} in version: {version.version_id}")